# Cross-Validation for Final Model: ResNet50V2 (Transfer Learning) — Skin Disease Classification

This notebook performs **Stratified K-Fold Cross-Validation** on the final selected model architecture
(ResNet50V2 backbone + custom classification head) to obtain a robust estimate of generalization
performance, rather than relying on a single train/val/test split.

For each fold, a **fresh copy of the model** (ImageNet-pretrained ResNet50V2, frozen backbone) is
trained from scratch on that fold's training portion and evaluated on its held-out validation portion.
Per-fold metrics (accuracy, precision, recall, F1, Cohen's Kappa, MCC, Jaccard) are collected and
summarised (mean ± std) at the end, along with per-class results and confusion matrices.


## 1. Setup

In [ ]:
!pip install -q scikit-learn


In [ ]:
import os
import time
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    cohen_kappa_score, matthews_corrcoef, jaccard_score
)

print("TensorFlow version:", tf.__version__)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Configuration

Update `BASE_DIR` / the three split folders below if your paths differ. All three splits
(`train`, `val`, `test`) are pooled together and re-split into K stratified folds, so that the
cross-validation estimate uses the full labelled dataset rather than just the original training split.


In [ ]:
# ---- Paths (same dataset used for the final model) ----
train_data = "/content/drive/MyDrive/defense/dataset_split/train"
valid_data = "/content/drive/MyDrive/defense/dataset_split/val"
test_data  = "/content/drive/MyDrive/defense/dataset_split/test"

# ---- Model / training config ----
IMG_SIZE   = (224, 224)
BATCH_SIZE = 64
K_FOLDS    = 5
EPOCHS     = 30           # per fold (cross-val is expensive; adjust as needed)
LR         = 0.001
SEED       = 42

# ---- Output locations ----
OUTPUT_DIR = "/content/drive/MyDrive/defense/save/cross_validation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.random.seed(SEED)
tf.random.set_seed(SEED)


## 3. Build a Single Pooled DataFrame (filepath, label)

We walk the three existing split directories and combine every image into one table, then let
`StratifiedKFold` do the splitting for cross-validation. Class names are taken from the `train`
directory (all splits share the same 12 classes).


In [ ]:
def collect_filepaths(directory):
    """Return a DataFrame with columns ['filepath', 'label'] for all images under `directory`,
    where `directory` contains one sub-folder per class."""
    directory = pathlib.Path(directory)
    records = []
    for class_dir in sorted([d for d in directory.iterdir() if d.is_dir()]):
        for img_path in class_dir.glob('*.jpg'):
            records.append({'filepath': str(img_path), 'label': class_dir.name})
    return pd.DataFrame(records)

df_train = collect_filepaths(train_data)
df_val   = collect_filepaths(valid_data)
df_test  = collect_filepaths(test_data)

df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)

print(f"Train images: {len(df_train)}")
print(f"Val images:   {len(df_val)}")
print(f"Test images:  {len(df_test)}")
print(f"Total pooled images for CV: {len(df_all)}")

class_names = sorted(df_all['label'].unique())
num_classes = len(class_names)
print(f"\nNumber of classes: {num_classes}")
print(class_names)

df_all['label'].value_counts()


## 4. Model Builder (final selected architecture)

Same architecture as the final selected model: frozen ImageNet ResNet50V2 backbone + GAP + Dense(256, relu) + Dense(128, relu) + Dense(64, relu) + Dense(num_classes, softmax).

In [ ]:
def build_model(num_classes, img_size=IMG_SIZE, lr=LR):
    """Builds and compiles a fresh instance of the final selected model."""
    base_model = tf.keras.applications.ResNet50V2(
        weights='imagenet', include_top=False, input_shape=img_size + (3,)
    )
    for layer in base_model.layers:
        layer.trainable = False

    input_tensor = Input(shape=img_size + (3,))
    x = base_model(input_tensor)
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dense(128, activation='relu')(x)
    x = Dense(64, activation='relu')(x)
    output = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=input_tensor, outputs=output)
    model.compile(
        loss='categorical_crossentropy',
        optimizer=Adam(learning_rate=lr),
        metrics=['accuracy']
    )
    return model


## 5. Stratified K-Fold Cross-Validation Loop

For each fold:
1. Split the pooled data into a training portion and a held-out validation portion (stratified by class).
2. Build fresh `ImageDataGenerator`s: augmentation on train, rescale-only on validation.
3. Build a **fresh** model (re-initialized head, ImageNet weights for the frozen backbone).
4. Train with early stopping + LR reduction, keeping the best checkpoint (by `val_accuracy`).
5. Evaluate on the held-out fold and store all metrics.


In [ ]:
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

fold_histories = []
fold_metrics = []
fold_reports = []
fold_confusion_matrices = []

X = df_all['filepath'].values
y = df_all['label'].values

overall_start = time.time()

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    print("\n" + "=" * 70)
    print(f"FOLD {fold_idx}/{K_FOLDS}")
    print("=" * 70)

    fold_train_df = df_all.iloc[train_idx].reset_index(drop=True)
    fold_val_df   = df_all.iloc[val_idx].reset_index(drop=True)
    print(f"Train samples: {len(fold_train_df)} | Val samples: {len(fold_val_df)}")

    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )
    val_datagen = ImageDataGenerator(rescale=1.0 / 255)

    train_gen = train_datagen.flow_from_dataframe(
        fold_train_df, x_col='filepath', y_col='label',
        target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', classes=class_names, shuffle=True, seed=SEED
    )
    val_gen = val_datagen.flow_from_dataframe(
        fold_val_df, x_col='filepath', y_col='label',
        target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', classes=class_names, shuffle=False
    )

    model = build_model(num_classes)

    fold_ckpt_path = os.path.join(OUTPUT_DIR, f"fold_{fold_idx}_best.weights.h5")
    callbacks = [
        ModelCheckpoint(fold_ckpt_path, save_best_only=True, monitor='val_accuracy',
                         save_weights_only=True, save_freq='epoch'),
        EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6)
    ]

    fold_start = time.time()
    history = model.fit(
        train_gen,
        steps_per_epoch=max(1, train_gen.samples // BATCH_SIZE),
        epochs=EPOCHS,
        validation_data=val_gen,
        validation_steps=max(1, val_gen.samples // BATCH_SIZE),
        callbacks=callbacks,
        verbose=1
    )
    fold_duration = time.time() - fold_start
    print(f"Fold {fold_idx} training time: {fold_duration:.1f}s")

    # Load best weights for this fold before evaluating
    model.load_weights(fold_ckpt_path)

    # ---- Evaluate on held-out fold ----
    val_gen.reset()
    pred_probs = model.predict(val_gen, steps=int(np.ceil(val_gen.samples / BATCH_SIZE)))
    y_pred = np.argmax(pred_probs, axis=1)
    y_true = val_gen.classes[: len(y_pred)]

    acc = accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    jaccard = jaccard_score(y_true, y_pred, average='micro')
    report_dict = classification_report(
        y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred)

    fold_metrics.append({
        'fold': fold_idx,
        'accuracy': acc,
        'precision_macro': report_dict['macro avg']['precision'],
        'recall_macro': report_dict['macro avg']['recall'],
        'f1_macro': report_dict['macro avg']['f1-score'],
        'cohen_kappa': kappa,
        'mcc': mcc,
        'jaccard_micro': jaccard,
        'train_time_sec': fold_duration,
        'best_epoch': int(np.argmin(history.history['val_loss']) + 1),
    })
    fold_reports.append(report_dict)
    fold_confusion_matrices.append(cm)
    fold_histories.append(history.history)

    print(f"Fold {fold_idx} -> Accuracy: {acc:.4f} | F1(macro): {report_dict['macro avg']['f1-score']:.4f} "
          f"| Kappa: {kappa:.4f} | MCC: {mcc:.4f}")

    # free memory before next fold
    tf.keras.backend.clear_session()

overall_duration = time.time() - overall_start
print(f"\nTotal cross-validation time: {overall_duration/60:.1f} minutes")


## 6. Per-Fold Results Table

In [ ]:
results_df = pd.DataFrame(fold_metrics)
results_df = results_df.set_index('fold')
results_df


In [ ]:
summary = results_df[['accuracy', 'precision_macro', 'recall_macro', 'f1_macro',
                       'cohen_kappa', 'mcc', 'jaccard_micro']].agg(['mean', 'std'])
print("Cross-Validation Summary (mean ± std across folds):\n")
for metric in summary.columns:
    print(f"{metric:>18s}: {summary.loc['mean', metric]:.4f} ± {summary.loc['std', metric]:.4f}")

summary


## 7. Visualise Fold Performance

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(results_df.index.astype(str), results_df['accuracy'], color='#4C72B0', label='Accuracy')
ax.axhline(results_df['accuracy'].mean(), color='red', linestyle='--',
           label=f"Mean accuracy = {results_df['accuracy'].mean():.4f}")
ax.set_xlabel('Fold')
ax.set_ylabel('Validation Accuracy')
ax.set_title(f'{K_FOLDS}-Fold Cross-Validation Accuracy — Final Selected Model (ResNet50V2)')
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cv_accuracy_per_fold.png'))
plt.show()


In [ ]:
metrics_to_plot = ['accuracy', 'f1_macro', 'cohen_kappa', 'mcc']
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(20, 4))
for ax, metric in zip(axes, metrics_to_plot):
    ax.plot(results_df.index, results_df[metric], marker='o')
    ax.axhline(results_df[metric].mean(), color='red', linestyle='--', alpha=0.7)
    ax.set_title(metric)
    ax.set_xlabel('Fold')
    ax.set_ylim(0, 1 if metric != 'mcc' else None)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cv_metrics_per_fold.png'))
plt.show()


In [ ]:
# Training curves for every fold (loss & accuracy)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, hist in enumerate(fold_histories, start=1):
    axes[0].plot(hist['loss'], label=f'Fold {i} train')
    axes[0].plot(hist['val_loss'], linestyle='--', label=f'Fold {i} val')
    axes[1].plot(hist['accuracy'], label=f'Fold {i} train')
    axes[1].plot(hist['val_accuracy'], linestyle='--', label=f'Fold {i} val')

axes[0].set_title('Loss per epoch (all folds)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[1].set_title('Accuracy per epoch (all folds)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[0].legend(fontsize=7, ncol=2)
axes[1].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cv_training_curves.png'))
plt.show()


## 8. Aggregated Confusion Matrix (summed across all folds)

In [ ]:
aggregated_cm = np.sum(fold_confusion_matrices, axis=0)

plt.figure(figsize=(9, 7))
sns.set(font_scale=1.0)
sns.heatmap(aggregated_cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title(f"Aggregated Confusion Matrix Across {K_FOLDS} Folds")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cv_aggregated_confusion_matrix.png'))
plt.show()


## 9. Per-Class Metrics Averaged Across Folds

In [ ]:
per_class_rows = []
for class_name in class_names:
    precisions = [rep[class_name]['precision'] for rep in fold_reports]
    recalls    = [rep[class_name]['recall'] for rep in fold_reports]
    f1s        = [rep[class_name]['f1-score'] for rep in fold_reports]
    supports   = [rep[class_name]['support'] for rep in fold_reports]
    per_class_rows.append({
        'class': class_name,
        'precision_mean': np.mean(precisions), 'precision_std': np.std(precisions),
        'recall_mean': np.mean(recalls), 'recall_std': np.std(recalls),
        'f1_mean': np.mean(f1s), 'f1_std': np.std(f1s),
        'avg_support_per_fold': np.mean(supports),
    })

per_class_df = pd.DataFrame(per_class_rows).set_index('class')
per_class_df


## 10. Save All Results

In [ ]:
results_df.to_csv(os.path.join(OUTPUT_DIR, 'cv_fold_metrics.csv'))
per_class_df.to_csv(os.path.join(OUTPUT_DIR, 'cv_per_class_metrics.csv'))
summary.to_csv(os.path.join(OUTPUT_DIR, 'cv_summary.csv'))

with open(os.path.join(OUTPUT_DIR, 'cv_config.json'), 'w') as f:
    json.dump({
        'img_size': IMG_SIZE, 'batch_size': BATCH_SIZE, 'k_folds': K_FOLDS,
        'epochs': EPOCHS, 'learning_rate': LR, 'seed': SEED,
        'num_classes': num_classes, 'class_names': class_names,
        'total_images': int(len(df_all)),
    }, f, indent=2)

print("Saved:")
print(f" - {OUTPUT_DIR}/cv_fold_metrics.csv")
print(f" - {OUTPUT_DIR}/cv_per_class_metrics.csv")
print(f" - {OUTPUT_DIR}/cv_summary.csv")
print(f" - {OUTPUT_DIR}/cv_config.json")
print(f" - Per-fold best weights: fold_<n>_best.weights.h5")


In [ ]:
# ============================================================
# DOWNLOAD ALL GENERATED CROSS-VALIDATION IMAGES
# Run this cell AFTER all previous cells are finished
# ============================================================

import os
import zipfile
from google.colab import files

# Folder where your notebook saves the figures
OUTPUT_DIR = "/content/drive/MyDrive/defense/save/cross_validation"

# Image files generated by the notebook
image_files = [
    "cv_accuracy_per_fold.png",
    "cv_metrics_per_fold.png",
    "cv_training_curves.png",
    "cv_aggregated_confusion_matrix.png"
]

# Create a ZIP file
zip_path = "/content/cross_validation_images.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:

    for filename in image_files:
        file_path = os.path.join(OUTPUT_DIR, filename)

        if os.path.exists(file_path):
            zipf.write(file_path, arcname=filename)
            print("Added:", filename)
        else:
            print("NOT FOUND:", filename)

# Download ZIP to your computer
if os.path.exists(zip_path):
    print("\nZIP file created successfully!")
    files.download(zip_path)
else:
    print("\nNo files were found.")

In [ ]:
# ============================================================
# FIND AND DOWNLOAD ALL CROSS-VALIDATION PLOTS
# ============================================================

import os
import zipfile
from google.colab import drive, files

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------
drive.mount('/content/drive', force_remount=True)

# ------------------------------------------------------------
# 2. Files we want
# ------------------------------------------------------------
target_files = [
    "cv_accuracy_per_fold.png",
    "cv_metrics_per_fold.png",
    "cv_training_curves.png",
    "cv_aggregated_confusion_matrix.png"
]

# ------------------------------------------------------------
# 3. Search entire Google Drive
# ------------------------------------------------------------
print("\nSearching Google Drive...")
print("=" * 60)

found_files = {}

for root, dirs, filenames in os.walk("/content/drive/MyDrive"):

    for filename in filenames:

        if filename in target_files:

            full_path = os.path.join(root, filename)

            # If duplicate files exist, keep the first one
            if filename not in found_files:
                found_files[filename] = full_path

# ------------------------------------------------------------
# 4. Show results
# ------------------------------------------------------------
print("\nSEARCH RESULTS")
print("=" * 60)

for filename in target_files:

    if filename in found_files:
        print("FOUND    :", filename)
        print("LOCATION :", found_files[filename])
        print()
    else:
        print("NOT FOUND:", filename)
        print()

# ------------------------------------------------------------
# 5. Create ZIP
# ------------------------------------------------------------
zip_path = "/content/cross_validation_plots.zip"

if len(found_files) > 0:

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:

        for filename, full_path in found_files.items():

            zipf.write(
                full_path,
                arcname=filename
            )

    print("=" * 60)
    print(f"Successfully added {len(found_files)} image(s) to ZIP.")
    print("ZIP:", zip_path)
    print("=" * 60)

    # --------------------------------------------------------
    # 6. Download
    # --------------------------------------------------------
    files.download(zip_path)

else:

    print("=" * 60)
    print("NO PLOT FILES WERE FOUND ANYWHERE IN MyDrive.")
    print("=" * 60)

    print("\nThis means the plot-generation cells probably")
    print("did not actually save the PNG files.")

## 11. Conclusion

Report the **mean ± standard deviation** of each metric across the K_FOLDS folds above as the
robustness estimate for the final selected ResNet50V2 model. A small standard deviation across
folds indicates the single train/val/test split result generalizes well; a large standard deviation
suggests the original split may have been favourable/unfavourable by chance and the mean CV
score should be reported instead of (or alongside) the single-split test accuracy.
